# 14 — Audio Understanding — Classification, Tagging & Beyond Speech

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/14_audio_understanding_classification_and_tagging.ipynb)

⏱ **Reading time:** ~30 min &nbsp;|&nbsp; **Prerequisites:** Notebooks 1–13

## Learning Objectives

By the end of this notebook you will be able to:

- **Visualise** audio in multiple representations — waveform, spectrogram, mel-spectrogram, and MFCCs — and explain when each is most useful.
- **Classify** arbitrary audio clips into hundreds of categories using the Audio Spectrogram Transformer (AST).
- **Tag** audio with multiple overlapping labels using sigmoid-based multi-label prediction.
- **Extract** dense audio embeddings and use them for similarity search and retrieval.
- **Analyse** prosodic features (pitch, energy, speaking rate) that carry emotional and paralinguistic information.
- **Design** a simple audio search engine backed by embedding-based nearest-neighbour retrieval.

## Audio Is More Than Speech

When we think of audio AI, speech recognition comes to mind first. But the sonic world is far richer. Music carries melody, rhythm, and timbre. Environmental sounds — rain, traffic, birdsong — provide context about a scene. Industrial equipment emits acoustic signatures that reveal mechanical health. A baby's cry, a glass breaking, a fire alarm — each triggers instant human recognition.

Humans excel at **auditory scene analysis**: separating overlapping sources (the cocktail party effect), localising sounds spatially, and recognising events from brief snippets. Replicating this in machines requires models that go beyond speech-to-text and learn general audio representations.

Compared to text (discrete tokens) and vision (spatial grids), audio is a **1-D temporal signal** with rich frequency content. A single second at 16 kHz contains 16,000 samples — far more raw values than a sentence of text, yet less spatial structure than an image. This unique character demands specialised representations and architectures.

## Audio Representations

Before a model can classify audio, the raw waveform must be transformed into a useful representation:

| Representation | Domain | Dimensionality | Typical Use |
|---|---|---|---|
| **Raw waveform** | Time | 1-D | End-to-end models (WaveNet) |
| **Spectrogram** | Time × Frequency | 2-D | General audio analysis |
| **Mel-spectrogram** | Time × Mel bands | 2-D | AST, Whisper, most modern models |
| **MFCCs** | Cepstral coefficients | 1-D per frame | Classical speech/speaker ID |
| **Learned embeddings** | Latent space | 1-D vector | Retrieval, clustering |

The **mel-spectrogram** has become the dominant input format. It applies the mel scale — a perceptual frequency mapping that mirrors human hearing — to a short-time Fourier transform. The result is a 2-D "image" of sound that vision-based architectures (like ViT) can process directly. This insight is the foundation of the Audio Spectrogram Transformer.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
!pip install -q transformers torch torchaudio datasets librosa bitsandbytes accelerate

import warnings, os
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
import librosa
import librosa.display
from IPython.display import Audio, display

print(f"torch  {torch.__version__}")
print(f"GPU    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Visualise four audio representations ──────────────────────────────────
SR = 16_000  # sample rate
DURATION = 3  # seconds

# Create a rich synthetic signal: chord + chirp + noise burst
t = np.linspace(0, DURATION, SR * DURATION, endpoint=False)
chord = 0.3 * np.sin(2 * np.pi * 261.6 * t)  # C4
chord += 0.3 * np.sin(2 * np.pi * 329.6 * t)  # E4
chord += 0.3 * np.sin(2 * np.pi * 392.0 * t)  # G4
chirp = 0.4 * np.sin(2 * np.pi * (200 + 2000 * t / DURATION) * t)
noise_burst = 0.2 * np.random.randn(len(t)) * (t > 1.5).astype(float) * (t < 2.0).astype(float)
y = (chord * (t < 1.5).astype(float) + chirp * (t >= 1.5).astype(float) + noise_burst).astype(np.float32)
y = y / np.max(np.abs(y))  # normalise

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# (a) Waveform
axes[0, 0].plot(t, y, linewidth=0.4, color="steelblue")
axes[0, 0].set_title("(a) Waveform"); axes[0, 0].set_xlabel("Time (s)"); axes[0, 0].set_ylabel("Amplitude")

# (b) Spectrogram
S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))
S_db = librosa.amplitude_to_db(S, ref=np.max)
librosa.display.specshow(S_db, sr=SR, hop_length=256, x_axis="time", y_axis="hz", ax=axes[0, 1], cmap="magma")
axes[0, 1].set_title("(b) Spectrogram")

# (c) Mel-spectrogram
M = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128, n_fft=1024, hop_length=256)
M_db = librosa.power_to_db(M, ref=np.max)
librosa.display.specshow(M_db, sr=SR, hop_length=256, x_axis="time", y_axis="mel", ax=axes[1, 0], cmap="magma")
axes[1, 0].set_title("(c) Mel-spectrogram")

# (d) MFCCs
mfccs = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=13, n_fft=1024, hop_length=256)
librosa.display.specshow(mfccs, sr=SR, hop_length=256, x_axis="time", ax=axes[1, 1], cmap="coolwarm")
axes[1, 1].set_title("(d) MFCCs"); axes[1, 1].set_ylabel("Coefficient")

plt.tight_layout()
plt.show()
print(f"Waveform samples: {len(y):,}  |  Spectrogram shape: {S.shape}  |  Mel shape: {M.shape}  |  MFCC shape: {mfccs.shape}")

## Audio Classification with the Audio Spectrogram Transformer (AST)

The **Audio Spectrogram Transformer** (Gong et al., 2021) is an elegant model that reframes audio classification as an image-recognition problem. The pipeline is:

1. Convert waveform → 128-band mel-spectrogram.
2. Split the spectrogram into fixed-size **patches** (typically 16 × 16), just like a Vision Transformer (ViT) splits an image.
3. Flatten and linearly project each patch into a token embedding.
4. Prepend a `[CLS]` token, add positional embeddings, and pass through a standard Transformer encoder.
5. The `[CLS]` representation feeds into a classification head over **527 AudioSet classes**.

AST achieves state-of-the-art performance on AudioSet (mean average precision 0.459) and transfers well to downstream tasks. Because it re-uses ViT machinery, it inherits decades of image-model engineering — pre-training, data augmentation (SpecAugment), and efficient attention variants all carry over directly.

The key insight is that a mel-spectrogram *is* an image: the x-axis is time, the y-axis is frequency, and pixel intensity encodes energy. Treating it as such lets us leverage the full power of vision transformers.

In [ ]:
# ── Load AST and classify audio ─────────────────────────────────────────
from transformers import pipeline

classifier = pipeline(
    "audio-classification",
    model="MIT/ast-finetuned-audioset-10-10-0.4593",
    device=0 if torch.cuda.is_available() else -1,
)

# Create a synthetic test signal — a 440 Hz sine (concert A)
sr = 16_000
duration = 5
t_test = np.linspace(0, duration, sr * duration, endpoint=False)
sine_440 = (0.5 * np.sin(2 * np.pi * 440 * t_test)).astype(np.float32)

preds = classifier(sine_440, top_k=5)

print("Top-5 predictions for a 440 Hz sine wave:")
print("-" * 45)
for p in preds:
    print(f"  {p['label']:30s}  {p['score']:.4f}")

### Understanding AudioSet Labels

AudioSet is a massive, hierarchically organised ontology of sound events curated by Google Research. It contains over **2 million** 10-second clips drawn from YouTube, each annotated with one or more labels from a taxonomy of **527 classes**.

The hierarchy spans from broad categories like *Music*, *Speech*, and *Animal* down to very specific labels such as *Tabla*, *Yodeling*, *Fire alarm*, or *Purr*. This granularity is both a strength and a challenge:

- **Strength:** Fine-grained labels enable precise sound-event detection in complex scenes.
- **Challenge:** Many labels are rare (long-tail distribution), and annotator agreement drops for subtle distinctions (e.g., *Whispering* vs. *Muttering*). Models may be highly confident on common categories yet unreliable on rare ones.

When interpreting AST predictions, keep in mind that the model was trained on YouTube audio, which skews towards music, speech, and popular sound effects. Industrial or domain-specific sounds may be under-represented.

In [ ]:
# ── Classify multiple synthetic signals and visualise ─────────────────
signals = {
    "Pure 440 Hz":    0.5 * np.sin(2 * np.pi * 440 * t_test),
    "Chord (C-E-G)":  0.3 * (np.sin(2*np.pi*261.6*t_test) + np.sin(2*np.pi*329.6*t_test) + np.sin(2*np.pi*392.0*t_test)),
    "White noise":     np.random.randn(len(t_test)) * 0.3,
    "Chirp 200-4k":   0.5 * np.sin(2*np.pi*(200 + 3800*t_test/duration)*t_test),
    "Silence":         np.zeros(len(t_test)),
}

rows = []
for name, sig in signals.items():
    sig = sig.astype(np.float32)
    top3 = classifier(sig, top_k=3)
    for rank, p in enumerate(top3, 1):
        rows.append({"Signal": name, "Rank": rank, "Label": p["label"], "Score": round(p["score"], 4)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Horizontal bar chart of top-1 predictions
top1 = df[df["Rank"] == 1].copy()
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(top1["Signal"], top1["Score"], color="teal")
for i, row in top1.iterrows():
    ax.text(row["Score"] + 0.01, row["Signal"], row["Label"], va="center", fontsize=9)
ax.set_xlim(0, 1.0); ax.set_xlabel("Confidence"); ax.set_title("Top-1 AST Predictions")
plt.tight_layout(); plt.show()

## Audio Tagging — Multiple Labels Per Clip

Classification asks *"what is the single best label?"* — it applies **softmax** over classes and picks the argmax. But real-world audio rarely contains just one sound. A street recording might simultaneously contain *Traffic noise*, *Car horn*, *Speech*, and *Bird*. This is the **multi-label tagging** setting.

In tagging, we replace the softmax with independent **sigmoid** activations per class. Each class gets its own probability in [0, 1], and we apply a threshold (typically 0.5) to decide which tags are present. The model was in fact trained with binary cross-entropy — a multi-label loss — so the raw logits already support this mode.

The distinction matters for deployment: a softmax classifier will always assign most probability mass to *something*, even when the clip is ambiguous. Sigmoid tagging allows the model to say "several things are happening at once" or even "nothing notable is present" if all probabilities fall below threshold.

In [ ]:
# ── Multi-label audio tagging with sigmoid thresholding ───────────────
from transformers import ASTForAudioClassification, ASTFeatureExtractor

model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_name)
model = ASTForAudioClassification.from_pretrained(model_name)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()

# Mix two signals: chord + noise burst (simulates multi-source audio)
mixed = 0.4 * np.sin(2*np.pi*440*t_test) + 0.3 * np.random.randn(len(t_test))
mixed = mixed.astype(np.float32)

inputs = feature_extractor(mixed, sampling_rate=16000, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.sigmoid(logits).cpu().squeeze().numpy()
threshold = 0.5
tag_indices = np.where(probs >= threshold)[0]

id2label = model.config.id2label
print(f"Tags above {threshold} threshold ({len(tag_indices)} found):")
print("-" * 45)
for idx in tag_indices:
    print(f"  {id2label[idx]:35s}  {probs[idx]:.4f}")

# Also show top-10 by probability regardless of threshold
top10 = np.argsort(probs)[::-1][:10]
print("\nTop-10 classes by probability:")
for idx in top10:
    marker = " \u2713" if probs[idx] >= threshold else ""
    print(f"  {id2label[idx]:35s}  {probs[idx]:.4f}{marker}")

## Audio Embeddings for Similarity Search

Beyond classification labels, the intermediate representations of a pre-trained audio model form a powerful **embedding space**. The `[CLS]` token from AST's final Transformer layer encodes a 768-dimensional summary of the entire audio clip — a dense vector that captures timbre, rhythm, pitch content, and semantic category all at once.

These embeddings enable a range of applications:

- **Audio deduplication** — find near-duplicate recordings even after re-encoding or volume changes.
- **Music recommendation** — retrieve songs with similar sonic characteristics.
- **Sound effect search** — a sound designer queries with one gunshot sample and retrieves all similar ones from a library.
- **Anomaly detection** — in industrial settings, an unusual embedding signals equipment malfunction.

The standard tool for comparing embeddings is **cosine similarity**: values near 1.0 indicate near-identical sounds; values near 0 suggest completely different acoustic content.

In [ ]:
# ── Extract embeddings and compute similarity matrix ──────────────────
from transformers import ASTModel
from torch.nn.functional import cosine_similarity

embed_model = ASTModel.from_pretrained(model_name)
embed_model.eval()
if torch.cuda.is_available():
    embed_model = embed_model.cuda()

def get_embedding(audio_array, sr=16000):
    """Extract [CLS] embedding from AST."""
    inputs = feature_extractor(audio_array, sampling_rate=sr, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = embed_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu()  # [CLS] token

# Build embeddings for our test signals
names = list(signals.keys())
embeddings = []
for name in names:
    emb = get_embedding(signals[name].astype(np.float32))
    embeddings.append(emb)
emb_tensor = torch.cat(embeddings, dim=0)  # (N, 768)

# Cosine similarity matrix
n = emb_tensor.shape[0]
sim_matrix = torch.zeros(n, n)
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(emb_tensor[i].unsqueeze(0), emb_tensor[j].unsqueeze(0))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix.numpy(), cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(names)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="Cosine Similarity")
ax.set_title("Audio Embedding Similarity Matrix")
plt.tight_layout(); plt.show()

## Building an Audio Search Engine

A practical audio search engine follows a simple two-phase architecture:

**Indexing phase (offline):**
1. Chunk each audio file into fixed-length segments (e.g., 5 or 10 seconds).
2. Extract an embedding for each segment using a pre-trained model like AST.
3. Store embeddings in a vector index (FAISS, Annoy, or a vector database like Pinecone).

**Query phase (online):**
1. The user provides a query audio clip (or a text description, if using a cross-modal model).
2. Extract the query embedding.
3. Perform nearest-neighbour lookup against the index.
4. Return the top-K most similar segments with metadata.

Practical considerations include index size (768-dim float32 vectors at ~3 KB each — a million clips requires ~3 GB), latency (approximate nearest-neighbour algorithms trade accuracy for speed), and the question of whether to use a single global embedding or multiple embeddings per clip for finer-grained matching.

In [ ]:
# ── Simple embedding-based audio search ───────────────────────────────
# Build a small "corpus" of audio clips
corpus = {
    "Piano C4":      0.5 * np.sin(2*np.pi*261.6*t_test),
    "Piano E4":      0.5 * np.sin(2*np.pi*329.6*t_test),
    "Piano G4":      0.5 * np.sin(2*np.pi*392.0*t_test),
    "High beep":     0.5 * np.sin(2*np.pi*2000*t_test),
    "Low hum":       0.5 * np.sin(2*np.pi*80*t_test),
    "White noise":   np.random.randn(len(t_test)) * 0.3,
    "Chirp":         0.5 * np.sin(2*np.pi*(100 + 5000*t_test/duration)*t_test),
    "Drum-like":     np.exp(-5*t_test) * np.sin(2*np.pi*150*t_test),
}

# Index: compute embeddings
corpus_names = list(corpus.keys())
corpus_embs = torch.cat([get_embedding(v.astype(np.float32)) for v in corpus.values()], dim=0)

def search(query_audio, top_k=3):
    """Return top-K most similar corpus items."""
    q_emb = get_embedding(query_audio)
    sims = cosine_similarity(q_emb, corpus_embs)  # (1, N) broadcast
    topk = sims.squeeze().argsort(descending=True)[:top_k]
    return [(corpus_names[i], sims[0, i].item()) for i in topk]

# Query: a tone close to E4 (330 Hz)
query = (0.5 * np.sin(2*np.pi*330*t_test)).astype(np.float32)
results = search(query, top_k=3)
print("Query: 330 Hz tone  \u2192  Nearest matches:")
for name, sim in results:
    print(f"  {name:20s}  similarity = {sim:.4f}")

# Query: noise
query_noise = (np.random.randn(len(t_test)) * 0.3).astype(np.float32)
results2 = search(query_noise, top_k=3)
print("\nQuery: white noise   \u2192  Nearest matches:")
for name, sim in results2:
    print(f"  {name:20s}  similarity = {sim:.4f}")

## Emotion Detection in Speech

Audio classification extends beyond *what* is said to *how* it is said. Prosodic features — pitch contour, energy, speaking rate, and voice quality — carry emotional information that text transcription loses entirely.

A sentence like "That's great" can express genuine enthusiasm or biting sarcasm depending on intonation. Detecting these cues requires models sensitive to **suprasegmental** features: patterns that span multiple phonemes or words rather than individual sounds.

Key challenges include cultural variation in emotional expression, context dependency, and the subtlety of real-world emotions versus the discrete categories (happy, sad, angry) used in most datasets.

In [ ]:
# ── Prosodic feature analysis ─────────────────────────────────────────
# Create synthetic "emotional" signals with different prosodic profiles
sr_p = 16_000
dur_p = 2.0
t_p = np.linspace(0, dur_p, int(sr_p * dur_p), endpoint=False)

# "Excited" — high pitch, rising, loud
excited = 0.8 * np.sin(2 * np.pi * (300 + 200 * t_p / dur_p) * t_p).astype(np.float32)
# "Calm" — low pitch, steady, quiet
calm = 0.2 * np.sin(2 * np.pi * 120 * t_p).astype(np.float32)
# "Sad" — low pitch, falling energy
sad = (0.5 * np.exp(-1.5 * t_p) * np.sin(2 * np.pi * 100 * t_p)).astype(np.float32)

styles = {"Excited": excited, "Calm": calm, "Sad": sad}

fig, axes = plt.subplots(3, 3, figsize=(14, 8))

for col, (label, sig) in enumerate(styles.items()):
    # Row 0: waveform
    axes[0, col].plot(t_p, sig, linewidth=0.5, color="steelblue")
    axes[0, col].set_title(f"{label} \u2014 Waveform"); axes[0, col].set_ylim(-1, 1)

    # Row 1: pitch (F0) via librosa
    f0, voiced, _ = librosa.pyin(sig, fmin=50, fmax=600, sr=sr_p)
    t_f0 = librosa.times_like(f0, sr=sr_p)
    axes[1, col].plot(t_f0, f0, color="darkorange", linewidth=1.2)
    axes[1, col].set_title(f"{label} \u2014 Pitch (F0)"); axes[1, col].set_ylabel("Hz")
    axes[1, col].set_ylim(0, 600)

    # Row 2: RMS energy
    rms = librosa.feature.rms(y=sig, frame_length=1024, hop_length=256)[0]
    t_rms = librosa.times_like(rms, sr=sr_p, hop_length=256)
    axes[2, col].plot(t_rms, rms, color="crimson", linewidth=1.2)
    axes[2, col].set_title(f"{label} \u2014 Energy (RMS)"); axes[2, col].set_ylabel("Amplitude")

for ax in axes.flat:
    ax.set_xlabel("Time (s)")
plt.tight_layout(); plt.show()

# Summary statistics
for label, sig in styles.items():
    f0, _, _ = librosa.pyin(sig, fmin=50, fmax=600, sr=sr_p)
    rms = librosa.feature.rms(y=sig)[0]
    f0_valid = f0[~np.isnan(f0)] if np.any(~np.isnan(f0)) else np.array([0])
    print(f"{label:10s}  mean F0={np.mean(f0_valid):.0f} Hz  std F0={np.std(f0_valid):.0f} Hz  mean RMS={np.mean(rms):.4f}")

## Exercises

### Exercise 1 — Confusion Analysis

Classify a batch of at least 10 different synthetic audio signals (varying frequencies, noise types, percussive vs. tonal) using the AST classifier. Collect top-3 predictions for each and build a confusion-style analysis: which AudioSet categories appear most often? Are there categories the model systematically confuses? Visualise your findings as a heatmap or bar chart.

In [ ]:
# ── Exercise 1 starter code ─────────────────────────────────────────
from collections import Counter

freqs = [60, 100, 200, 440, 800, 1000, 2000, 4000, 6000, 8000]
test_signals = {f"{f}Hz": (0.5 * np.sin(2*np.pi*f*t_test)).astype(np.float32) for f in freqs}

label_counter = Counter()
for name, sig in test_signals.items():
    preds = classifier(sig, top_k=3)
    for p in preds:
        label_counter[p["label"]] += 1

# TODO: visualise the most common predicted labels
# Hint: use label_counter.most_common(15) and a horizontal bar chart
print("Most common predicted labels:")
for label, count in label_counter.most_common(10):
    print(f"  {label:30s}  appeared {count} times")

### Exercise 2 — Audio Similarity Search Evaluation

Expand the corpus to at least 15 clips. For each clip, use it as a query and check whether the most similar result is semantically reasonable (e.g., two tones at nearby frequencies should rank close). Compute **precision@3** by manually labelling which retrievals are relevant. Report the average precision.

In [ ]:
# ── Exercise 2 starter code ─────────────────────────────────────────
# Expand the corpus
extended_corpus = dict(corpus)  # start from existing 8 clips
# TODO: add more signals, e.g.:
# extended_corpus["AM tone"] = (0.5 * (1 + 0.5*np.sin(2*np.pi*5*t_test)) * np.sin(2*np.pi*440*t_test)).astype(np.float32)
# extended_corpus["Harmonic stack"] = ...

# TODO: re-index, run leave-one-out search, evaluate precision@3
precision_scores = []
# for name in extended_corpus:
#     results = search_excluding(name, top_k=3)
#     relevant = count_relevant(name, results)
#     precision_scores.append(relevant / 3)
# print(f"Mean precision@3: {np.mean(precision_scores):.2f}")

### Exercise 3 — Layer-wise Embedding Comparison

AST has 12 Transformer layers. Extract the `[CLS]` token from layers 1, 6, and 12 (using `output_hidden_states=True`). For each layer, compute the similarity matrix over the test signals. Do early layers capture different structure than late layers? Which layer works best for nearest-neighbour classification?

In [ ]:
# ── Exercise 3 starter code ─────────────────────────────────────────
def get_layer_embedding(audio_array, layer_idx, sr=16000):
    """Extract [CLS] embedding from a specific Transformer layer."""
    inputs = feature_extractor(audio_array, sampling_rate=sr, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = embed_model(**inputs, output_hidden_states=True)
    # outputs.hidden_states is a tuple of (n_layers + 1) tensors
    return outputs.hidden_states[layer_idx][:, 0, :].cpu()

# TODO: extract embeddings from layers 1, 6, 12
# TODO: compute similarity matrices for each
# TODO: visualise side-by-side and compare

In [ ]:
# ── Batch classification helper ───────────────────────────────────────
def classify_batch(audio_list, names, top_k=3):
    """Classify a list of audio arrays and return a DataFrame."""
    rows = []
    for name, audio in zip(names, audio_list):
        preds = classifier(audio.astype(np.float32), top_k=top_k)
        for rank, p in enumerate(preds, 1):
            rows.append({"Audio": name, "Rank": rank, "Label": p["label"], "Score": p["score"]})
    return pd.DataFrame(rows)

# Demo: classify a few more sounds
extra_signals = [
    0.5 * np.sin(2*np.pi*1000*t_test),              # 1 kHz tone
    np.random.randn(len(t_test)) * 0.1,              # quiet noise
    np.exp(-3*t_test) * np.sin(2*np.pi*200*t_test),  # percussive
]
extra_names = ["1 kHz tone", "Quiet noise", "Percussive hit"]
df_extra = classify_batch(extra_signals, extra_names)
print(df_extra.to_string(index=False))

In [ ]:
# ── Embedding dimensionality and storage estimate ─────────────────────
emb_dim = emb_tensor.shape[1]
bytes_per_emb = emb_dim * 4  # float32
n_clips_1M = 1_000_000

print(f"Embedding dimensionality: {emb_dim}")
print(f"Bytes per embedding:      {bytes_per_emb:,} ({bytes_per_emb/1024:.1f} KB)")
print(f"Storage for 1M clips:     {n_clips_1M * bytes_per_emb / 1e9:.2f} GB")
print(f"Storage for 10M clips:    {10 * n_clips_1M * bytes_per_emb / 1e9:.2f} GB")

### Practical Tips for Audio Pipelines

- **Resampling:** Always resample to the model's expected rate (16 kHz) *before* feature extraction.
- **Segment length:** AST expects ~10-second clips. Window longer audio; zero-pad shorter audio.
- **Normalisation:** Peak-normalise or RMS-normalise to avoid volume-dependent predictions.

## Key Takeaways

| # | Insight |
|---|---|
| 1 | Audio is far richer than speech alone — environmental sounds, music, animal calls, and machine noises all carry structured information that models can learn. |
| 2 | Mel-spectrograms are the dominant input representation for modern audio models because they balance perceptual relevance with spatial structure (making them compatible with vision architectures). |
| 3 | AST treats a mel-spectrogram as an image and applies ViT-style patch embedding, achieving state-of-the-art classification on AudioSet's 527 classes. |
| 4 | Multi-label tagging with sigmoid outputs is more realistic than single-label classification for real-world audio, which often contains overlapping sound events. |
| 5 | Audio embeddings from pre-trained models enable similarity search, deduplication, and recommendation without task-specific fine-tuning. |
| 6 | Prosodic features (pitch, energy, rate) carry emotional and paralinguistic information that complements text-based analysis. |

## References

1. Gong, Y., Chung, Y.-A., & Glass, J. (2021). *AST: Audio Spectrogram Transformer*. [arXiv:2104.01778](https://arxiv.org/abs/2104.01778)
2. Gemmeke, J. F. et al. (2017). *Audio Set: An ontology and human-labeled dataset for audio events*. [ICASSP 2017](https://research.google.com/audioset/)
3. HuggingFace — [MIT/ast-finetuned-audioset](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593)

---

⬅️ [**Notebook 13 — Speech Recognition & Transcription**](13_speech_recognition_and_transcription.ipynb) &nbsp;|&nbsp; ➡️ [**Notebook 15 — Speech + Document Workflows**](15_speech_plus_document_workflows.ipynb)

This notebook extends the speech foundations from Notebook 13 into the broader world of non-speech audio understanding. Next, in Notebook 15, we will combine speech and document modalities into unified workflows.